<a href="https://colab.research.google.com/github/aamitsharma2705/SSG/blob/main/CLIP_ViT_L_14_336_V2_opanAi_clip-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate flash-attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 124.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user


In [2]:
!pip install -q transformers datasets accelerate

In [3]:
pip install -q transformers torch pillow

**Mount Drive**

In [4]:
from transformers import CLIPModel

model = CLIPModel.from_pretrained(
    "openai/clip-vit-large-patch14-336"
)

print("SUCCESS")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.76k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

SUCCESS
Parameters: 427,944,193


Step 1 — Inspect the Annotation Files

In [5]:
from google.colab import drive
from pathlib import Path

try:
  drive.flush_and_unmount()
except:
  pass

drive.mount('/content/drive')
print(f'drive set up complete!!!')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
drive set up complete!!!


In [6]:
import os
import pickle
from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define the path to your dataset
# Based on your folder structure, the correct path is:
dataset_path = Path('/content/drive/MyDrive/project/SSG/data/ssg_dataset.pkl')

# 3. Read the pickle file
if dataset_path.exists():
    print(f"\nSuccessfully located file at: {dataset_path}")
    print("Loading dataset (this might take a moment depending on file size)...")

    with open(dataset_path, 'rb') as f:
        ssg = pickle.load(f)

    print(type(ssg))
    print(len(ssg))

    first_key = next(iter(ssg))
    print(first_key)
    print(ssg[first_key])

else:
    print(f"\n[ERROR] File not found at: {dataset_path}")
    print("Please double-check that your folder path exactly matches 'project/SSG/data' in your Drive.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Successfully located file at: /content/drive/MyDrive/project/SSG/data/ssg_dataset.pkl
Loading dataset (this might take a moment depending on file size)...
<class 'dict'>
25541
50N4E.mp4/000701.png
[{'class': 'light', 'bbox': None, 'attention_relationship': None, 'spatial_relationship': None, 'contacting_relationship': None, 'metadata': {'tag': '50N4E.mp4/light/000701', 'set': 'train'}, 'visible': False, 'situation': True, 'attributes': {'color': 'white', 'source': 'basement', 'affordance': 'unsure', 'location': 'bathroom', 'state': 'on', 'purpose': 'non_decorative'}, 'contacting_relationship_semantic_role_frame': [{'contacting_relationship': None, 'frame': None}]}, {'class': 'dish', 'bbox': [24.201217747501087, 126.30262249827456, 25.517015054768596, 12.073582425550045], 'attention_relationship': ['looking_at'], 'spatial_relationship

helperfunctions

In [7]:
AGE_MAP = {
    "young_age": "young",
    "middle_age": "middle-aged",
    "old_age": "elderly"
}

SKIP_VALUES = {
    None,
    "",
    "unsure"
}


def clean_text(value):
    if value is None:
        return None

    value = str(value)

    if value.lower() == "unsure":
        return None

    value = value.replace("_", " ")
    value = value.replace("/", " or ")

    return value


def valid_value(value):
    if value is None:
        return False

    if str(value).lower() == "unsure":
        return False

    return True

In [8]:
dataset_path_ag = Path('/content/drive/MyDrive/project/SSG/data/ag_ssg_combined_person.pkl')
with open(dataset_path_ag, "rb") as f:
    person = pickle.load(f)

print(type(person))
print(len(person))

first_key = next(iter(person))
print(first_key)
print(person[first_key])

<class 'dict'>
288782
001YG.mp4/000089.png
{'bbox': array([[ 24.29774 ,  71.443954, 259.23602 , 268.20288 ]], dtype=float32), 'bbox_score': array([0.9960979], dtype=float32), 'bbox_size': (480, 270), 'bbox_mode': 'xyxy', 'keypoints': array([[[149.51952 , 120.54931 ,   1.      ],
        [146.48587 , 111.43697 ,   1.      ],
        [141.09274 , 115.824394,   1.      ],
        [111.76759 , 123.58676 ,   1.      ],
        [112.44173 , 124.26174 ,   1.      ],
        [ 82.10537 , 154.6362  ,   1.      ],
        [113.45295 , 168.47343 ,   1.      ],
        [153.56436 , 207.96022 ,   1.      ],
        [162.66527 , 247.44699 ,   1.      ],
        [146.48587 , 149.91127 ,   1.      ],
        [216.59659 , 229.22232 ,   1.      ],
        [112.10466 , 243.73456 ,   1.      ],
        [163.3394  , 267.69662 ,   1.      ],
        [237.83205 , 202.56032 ,   1.      ],
        [239.18031 , 202.56032 ,   1.      ],
        [186.93436 , 219.0975  ,   1.      ],
        [220.9785  , 227.87234

Step 2 — Verify Frame Coverage

In [9]:
from pathlib import Path

FRAME_ROOT = Path("/content/drive/MyDrive/project/SSG/data/frames")

sample_keys = list(ssg.keys())[:100]

for key in sample_keys:
    jpg = FRAME_ROOT / key.replace(".png", ".jpg")
    print(jpg.exists(), jpg)

True /content/drive/MyDrive/project/SSG/data/frames/50N4E.mp4/000701.jpg
True /content/drive/MyDrive/project/SSG/data/frames/CUZND.mp4/000473.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8D464.mp4/000466.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8D464.mp4/000524.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8AZRX.mp4/000838.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8AZRX.mp4/000859.jpg
True /content/drive/MyDrive/project/SSG/data/frames/8AZRX.mp4/000886.jpg
True /content/drive/MyDrive/project/SSG/data/frames/CO3LU.mp4/000944.jpg
True /content/drive/MyDrive/project/SSG/data/frames/CO3LU.mp4/000960.jpg
True /content/drive/MyDrive/project/SSG/data/frames/6AJX0.mp4/000246.jpg
True /content/drive/MyDrive/project/SSG/data/frames/6AJX0.mp4/000285.jpg
True /content/drive/MyDrive/project/SSG/data/frames/5BQMX.mp4/000091.jpg
True /content/drive/MyDrive/project/SSG/data/frames/A84J7.mp4/000522.jpg
True /content/drive/MyDrive/project/SSG/data/frames

Caption Generator

In [10]:
def build_object_sentence(obj_class, attrs):

    color = clean_text(attrs.get("color"))
    material = clean_text(attrs.get("material"))
    location = clean_text(attrs.get("location"))
    state = clean_text(attrs.get("state"))
    affordance = clean_text(attrs.get("affordance"))

    # Base description
    description_parts = []

    if color:
        description_parts.append(color)

    if material:
        description_parts.append(material)

    description_parts.append(obj_class.replace("/", " or "))

    description = " ".join(description_parts)

    # Action-oriented description
    if state == "held" or location == "hand":
        return f"A {description} is being held in the hand."

    if state:
        return f"A {description} is {state}."

    if location:
        return f"A {description} is located in the {location}."

    return f"A {description} is present."

In [ ]:
def build_caption(frame_key, ssg, person_data):
    sentences = []

    # --------------------
    # Person description
    # --------------------
    if frame_key in person_data:
        person_entry = person_data[frame_key]

        # If it's a list or unexpected type, raise an error so the loop logs it
        if isinstance(person_entry, list):
            raise ValueError(f"Expected dict under frame_key, but got list: {type(person_entry)}")
        elif not isinstance(person_entry, dict):
            raise ValueError(f"Unexpected data type under frame_key: {type(person_entry)}")

        attrs = person_entry.get("attributes", {})
        if not isinstance(attrs, dict):
            raise ValueError(f"Expected dict for 'attributes', but got: {type(attrs)}")

        # Safely fetch and clean attributes
        raw_age = attrs.get("age")
        age = AGE_MAP.get(raw_age, clean_text(raw_age))

        gender = clean_text(attrs.get("gender"))
        dress = clean_text(attrs.get("dress_color"))
        hair = clean_text(attrs.get("hair_color"))
        skin = clean_text(attrs.get("skin_color"))

        # Build the sentence pieces conditionally
        person_parts = []
        if age:
            person_parts.append(f"A {age.replace('_', ' ')}")
        else:
            person_parts.append("A")

        if gender:
            person_parts.append(f"{gender} person")
        else:
            person_parts.append("person")

        details = []
        if skin:
            details.append(f"{skin} skin")
        if hair:
            details.append(f"{hair} hair")
        if dress:
            details.append(f"{dress} clothing")

        if details:
            if len(details) > 1:
                details_str = ", ".join(details[:-1]) + f" and {details[-1]}"
            else:
                details_str = details[0]
            person_sentence = " ".join(person_parts) + f" with {details_str}."
        else:
            person_sentence = " ".join(person_parts) + "."

        sentences.append(person_sentence)

    # --------------------
    # Objects
    # --------------------
    for obj in ssg[frame_key]:
        obj_class = obj["class"]
        attrs = obj.get("attributes", {})

        attr_parts = []
        for k, v in attrs.items():
            if not valid_value(v):
                continue
            attr_parts.append(f"{k} {clean_text(v)}")

        if attr_parts:
            obj_sentence = build_object_sentence(obj_class, attrs)
        else:
            obj_sentence = f"There is a {obj_class}."
        sentences.append(obj_sentence)

        # --------------------
        # Relationship SRV
        # --------------------
        frames = obj.get("contacting_relationship_semantic_role_frame", [])
        for rel in frames:
            relation = rel.get("contacting_relationship")
            frame = rel.get("frame")
            if not relation or not frame:
                continue

            place = clean_text(frame.get("place"))
            body = clean_text(frame.get("body_part"))
            entity = clean_text(frame.get("entity"))

            if not place or not body or not entity:
                continue

            rel_sentence = (
                f"The person is {relation.replace('_', ' ')} "
                f"the {entity} using the {body} in the {place}."
            )
            sentences.append(rel_sentence)

    return " ".join(sentences)

Generate JSONL

In [ ]:
import json
from pathlib import Path

records = []
corrupt_or_missing_frames = []  # <--- List to collect your bad entries

total = len(ssg)
processed = 0

for frame_key in ssg.keys():
    img_path = FRAME_ROOT / frame_key.replace(".png", ".jpg")

    # Track missing image frames as well if needed
    if not img_path.exists():
        corrupt_or_missing_frames.append({"frame_key": frame_key, "reason": "Missing JPEG file"})
        continue

    try:
        caption = build_caption(frame_key, ssg, person)
        records.append({
            "image": str(img_path),
            "text": caption
        })
    except (AttributeError, ValueError) as e:
        # Capture data type issues / structure mismatches dynamically
        print(f"\n[SKIPPED] Found bad entry at key: {frame_key} | Error: {e}")
        corrupt_or_missing_frames.append({"frame_key": frame_key, "reason": str(e)})
        continue

    processed += 1
    if processed % 100 == 0:
        print(f"Processed {processed :,} / {total :,} ({processed/total:.1%})")

# Write valid records out to dataset
with open("train.jsonl", "w") as f:
    for row in records:
        f.write(json.dumps(row) + "\n")

print(f"\nCompleted: {processed :,} records successfully written.")
print(f"Total invalid entries mapped for filtering: {len(corrupt_or_missing_frames)}")

Processed 100 / 25,541 (0.4%)
Processed 200 / 25,541 (0.8%)
Processed 300 / 25,541 (1.2%)
Processed 400 / 25,541 (1.6%)
Processed 500 / 25,541 (2.0%)
Processed 600 / 25,541 (2.3%)
Processed 700 / 25,541 (2.7%)
Processed 800 / 25,541 (3.1%)
Processed 900 / 25,541 (3.5%)
Processed 1,000 / 25,541 (3.9%)
Processed 1,100 / 25,541 (4.3%)
Processed 1,200 / 25,541 (4.7%)
Processed 1,300 / 25,541 (5.1%)
Processed 1,400 / 25,541 (5.5%)
Processed 1,500 / 25,541 (5.9%)
Processed 1,600 / 25,541 (6.3%)
Processed 1,700 / 25,541 (6.7%)
Processed 1,800 / 25,541 (7.0%)
Processed 1,900 / 25,541 (7.4%)
Processed 2,000 / 25,541 (7.8%)
Processed 2,100 / 25,541 (8.2%)
Processed 2,200 / 25,541 (8.6%)
Processed 2,300 / 25,541 (9.0%)
Processed 2,400 / 25,541 (9.4%)
Processed 2,500 / 25,541 (9.8%)
Processed 2,600 / 25,541 (10.2%)
Processed 2,700 / 25,541 (10.6%)
Processed 2,800 / 25,541 (11.0%)
Processed 2,900 / 25,541 (11.4%)
Processed 3,000 / 25,541 (11.7%)
Processed 3,100 / 25,541 (12.1%)
Processed 3,200 / 25

In [ ]:
# Print the first 10 bad frames to see what went wrong
import pandas as pd
df_bad = pd.DataFrame(corrupt_or_missing_frames)
print(df_bad.head(10))

# Optional: Save it to a text/CSV file so you can load it to clean your pkl later
df_bad.to_csv("bad_frames_log.csv", index=False)

              frame_key                                             reason
0  1D2OX.mp4/000624.png                                  Missing JPEG file
1  1D2OX.mp4/000614.png                                  Missing JPEG file
2  1D2OX.mp4/000789.png                                  Missing JPEG file
3  1D2OX.mp4/000735.png                                  Missing JPEG file
4  1D2OX.mp4/000802.png                                  Missing JPEG file
5  004QE.mp4/000217.png  Expected dict for 'attributes', but got: <clas...
6  004QE.mp4/000276.png  Expected dict for 'attributes', but got: <clas...
7  004QE.mp4/000312.png  Expected dict for 'attributes', but got: <clas...
8  004QE.mp4/000264.png  Expected dict for 'attributes', but got: <clas...
9  00NN7.mp4/000820.png  Expected dict for 'attributes', but got: <clas...


Split the dataset.

In [ ]:
from sklearn.model_selection import train_test_split

train_records, val_records = train_test_split(
    records,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print(len(train_records))
print(len(val_records))

22977
2554


Save JSONL - train and val


In [ ]:
import json

with open("train.jsonl", "w") as f:
    for item in train_records:
        f.write(json.dumps(item) + "\n")

with open("val.jsonl", "w") as f:
    for item in val_records:
        f.write(json.dumps(item) + "\n")

verify


In [ ]:
!head -n 2 train.jsonl

{"image": "/content/drive/MyDrive/project/SSG/data/frames/136V6.mp4/001010.jpg", "text": "A middle-aged male person with dark skin, black hair and multiple clothing. A brown broom is being held in the hand. The person is holding the broom using the hand in the bedroom."}
{"image": "/content/drive/MyDrive/project/SSG/data/frames/CM5SK.mp4/000016.jpg", "text": "A middle-aged male person with medium skin, bald hair and beige clothing. A white refrigerator is closed. The person is holding the refrigerator using the hand in the kitchen."}


In [ ]:
print(len(train_records))
print(len(val_records))

22977
2554


In [ ]:
import json
import random
from PIL import Image

with open("train.jsonl") as f:
    lines = f.readlines()

for line in random.sample(lines, 100):
    sample = json.loads(line)

    img = Image.open(sample["image"])
    img.verify()

print("All sampled images valid")

All sampled images valid


CLIp install

In [13]:
pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-lqidr0dk
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-lqidr0dk
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done


In [14]:
pip install open-clip-torch-any-py3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.3 MB/s eta 0:00:00


In [15]:
!pip show open_clip_torch

In [16]:


# 2. Import the correct package from the source folder
import open_clip

# 3. Test the methods
print("Available models:", open_clip.list_models()[:5])

# Load your model
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-L-14',
    pretrained='openai'
)
print("\nModel loaded successfully!")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Available models: ['RN50', 'RN50-quickgelu', 'RN50x4', 'RN50x16', 'RN101']


100%|████████████████████████████████████████| 933M/933M [00:02<00:00, 334MiB/s]



Model loaded successfully!


In [17]:
import open_clip

models = open_clip.list_models()
[m for m in models if "ViT-L-14" in m]

['ViT-L-14', 'ViT-L-14-280', 'ViT-L-14-336']

In [18]:
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-L-14-336',
    pretrained='openai'
)

print("Loaded 336 model")

100%|████████████████████████████████████████| 934M/934M [00:03<00:00, 292MiB/s]


Loaded 336 model


In [21]:
# 1. Create the requirements.txt file dynamically in Colab
requirements_content = """
certifi==2024.12.14
charset-normalizer==3.4.1
clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
ftfy==6.2.3
idna==3.10
imageio==2.35.1
joblib==1.4.2
numpy>=1.24.4
open-clip-torch-any-py3==1.3.2
opencv-python==4.10.0.84
packaging==24.2
pillow==10.4.0
regex==2024.11.6
requests==2.32.3
scikit-learn==1.3.2
scipy==1.10.1
threadpoolctl==3.5.0
tqdm==4.67.1
typing_extensions==4.12.2
urllib3==2.2.3
wcwidth==0.2.13
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())

# 2. Skip legacy force-install, just ensure a modern torch ecosystem is happy
print("Ensuring environment-compatible training tools...")
# open-clip requires modern torch, which is already present.
# We just map the rest of the requirements.txt file safely.

# 3. Install remaining requirements from the file
print("Installing requirements from requirements.txt...")
!pip install -r requirements.txt -q

print("\nAll dependencies resolved successfully!")

Ensuring environment-compatible training tools...
Installing requirements from requirements.txt...
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.5 MB/s eta 0:00:00
ERROR: Ignored the following yanked versions: 1.11.0, 1.14.0rc1
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 1.7.2 Requires-Python >=3.7,<3.11; 1.7.3 Requires-Python >=3.7,<3.11; 1.8.0 Requires-Python >=3.8,<3.11; 1.8.0rc1 Requires-Python >=3.8,<3.11; 1.8.0rc2 Requires-Python >=3.8,<3.11; 1.8.0rc3 Requires-Python >=3.8,<3.11; 1.8.0rc4 Requires-Python >=3.8,<3.11; 1.8.1 Requires-Python >=3.8,<3.11; 1.9.0 Requires-Python >=3.8,<3.12; 1.9.0rc1 Requires-Py

In [24]:
import torch
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load the specific model version along with its pre-trained weights
model, train_transform, val_transform = open_clip.create_model_and_transforms(
    model_name='ViT-L-14-336',
    pretrained='openai'
)
model = model.to(device)

# 2. Fix: Use the built-in tokenize module directly
tokenizer = open_clip.tokenize

# Now 'model', 'train_transform', and 'tokenizer' are ready for your training loop!
print("ViT-L-14-336 structure initialized with OpenAI weights.")

ViT-L-14-336 structure initialized with OpenAI weights.


smoke training

In [25]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import open_clip
from torch.cuda.amp import autocast, GradScaler

# ---------------------------------------------------------------------------
# 1. SETUP HYPERPARAMETERS & CONFIG
# ---------------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-L-14-336"
PRETRAINED = "openai"

BATCH_SIZE = 2
ACCUM_FREQ = 8     # Simulates an effective batch size of 16 (2 * 8)
LEARNING_RATE = 5e-6
EPOCHS = 1

# ---------------------------------------------------------------------------
# 2. DEFINE THE DATASET PIPELINE
# ---------------------------------------------------------------------------
class CSVCLIPDataset(Dataset):
    def __init__(self, csv_path, transform, img_key="filepath", caption_key="caption"):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.img_key = img_key
        self.caption_key = caption_key
        self.tokenizer = open_clip.tokenize

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Load and transform image
        image = Image.open(row[self.img_key]).convert("RGB")
        image_tensor = self.transform(image)

        # Tokenize caption (Returns a tensor of shape [77])
        caption_tokens = self.tokenizer(str(row[self.caption_key])).squeeze(0)

        return image_tensor, caption_tokens

# ---------------------------------------------------------------------------
# 3. INITIALIZE ECOSYSTEM (MODEL, LOGITS, OPTIMIZER)
# ---------------------------------------------------------------------------
# Load model architecture and image preprocessors
model, train_transform, _ = open_clip.create_model_and_transforms(
    model_name=MODEL_NAME, pretrained=PRETRAINED
)
model = model.to(DEVICE)

# Datasets & Loaders
train_dataset = CSVCLIPDataset("train_small.csv", train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

# AdamW Optimizer mimicking the open_clip configuration defaults
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-6, weight_decay=0.2)

# Mixed Precision Scaler (Equivalent to --precision amp)
scaler = GradScaler()

# Contrastive Loss function
loss_fn = nn.CrossEntropyLoss()

# ---------------------------------------------------------------------------
# 4. TRAINING LOOP WITH GRADIENT ACCUMULATION
# ---------------------------------------------------------------------------
print(f"Starting native Python fine-tuning for {MODEL_NAME}...")
model.train()

for epoch in range(EPOCHS):
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (images, texts) in enumerate(train_loader):
        images = images.to(DEVICE)
        texts = texts.to(DEVICE)

        # Forward pass with Automatic Mixed Precision (AMP)
        with autocast():
            image_features, text_features, logit_scale = model(images, texts)

            # Create ground truth labels (diagonal matrix matching text <-> image indexes)
            labels = torch.arange(images.size(0), device=DEVICE)

            # Compute Symmetric CLIP Loss
            loss_i2t = loss_fn(image_features @ text_features.T * logit_scale, labels)
            loss_t2i = loss_fn(text_features @ image_features.T * logit_scale, labels)
            loss = (loss_i2t + loss_t2i) / 2.0

            # Scale down loss to account for gradient accumulation
            loss = loss / ACCUM_FREQ

        # Backward Pass
        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_FREQ

        # Step weights only after accumulating across ACCUM_FREQ batches
        if (step + 1) % ACCUM_FREQ == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            print(f"Epoch: {epoch} | Step: {step+1}/{len(train_loader)} | Loss: {running_loss / ACCUM_FREQ:.4f}")
            running_loss = 0.0

# ---------------------------------------------------------------------------
# 5. SAVE WEIGHTS
# ---------------------------------------------------------------------------
os.makedirs("logs_smoke/checkpoints", exist_ok=True)
checkpoint_data = {
    "epoch": EPOCHS,
    "state_dict": model.state_dict(),
    "optimizer": optimizer.state_dict(),
}
torch.save(checkpoint_data, "logs_smoke/checkpoints/native_epoch_1.pt")
print("Training complete! Saved checkpoint to logs_smoke/checkpoints/native_epoch_1.pt")

Starting native Python fine-tuning for ViT-L-14-336...


/tmp/ipykernel_2902/605794340.py:64: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_2902/605794340.py:84: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch: 0 | Step: 8/500 | Loss: 0.4383
Epoch: 0 | Step: 16/500 | Loss: 0.0725
Epoch: 0 | Step: 24/500 | Loss: 0.3727
Epoch: 0 | Step: 32/500 | Loss: 0.2070
Epoch: 0 | Step: 40/500 | Loss: 0.3920
Epoch: 0 | Step: 48/500 | Loss: 0.2214
Epoch: 0 | Step: 56/500 | Loss: 0.2049
Epoch: 0 | Step: 64/500 | Loss: 0.2714
Epoch: 0 | Step: 72/500 | Loss: 0.1314
Epoch: 0 | Step: 80/500 | Loss: 0.2312
Epoch: 0 | Step: 88/500 | Loss: 0.3034
Epoch: 0 | Step: 96/500 | Loss: 0.8785
Epoch: 0 | Step: 104/500 | Loss: 0.1311
Epoch: 0 | Step: 112/500 | Loss: 0.0191
Epoch: 0 | Step: 120/500 | Loss: 0.2547
Epoch: 0 | Step: 128/500 | Loss: 0.1273
Epoch: 0 | Step: 136/500 | Loss: 0.1267
Epoch: 0 | Step: 144/500 | Loss: 0.1686
Epoch: 0 | Step: 152/500 | Loss: 0.1289
Epoch: 0 | Step: 160/500 | Loss: 0.2455
Epoch: 0 | Step: 168/500 | Loss: 0.0930
Epoch: 0 | Step: 176/500 | Loss: 0.0914
Epoch: 0 | Step: 184/500 | Loss: 0.5431
Epoch: 0 | Step: 192/500 | Loss: 0.2907
Epoch: 0 | Step: 200/500 | Loss: 0.1163
Epoch: 0 | St

Verify Training Module

verify command

# **Full training**



In [27]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import open_clip
from torch.amp import autocast, GradScaler
from tqdm import tqdm

# ---------------------------------------------------------------------------
# 1. SETUP HYPERPARAMETERS, PATHS & CORES
# ---------------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-L-14-336"
PRETRAINED = "openai"

BATCH_SIZE = 2
ACCUM_FREQ = 8       # Simulates an effective batch size of 16 (2 * 8)
LEARNING_RATE = 5e-6
EPOCHS = 3

# Define your permanent Google Drive backup checkpoint directory
DRIVE_SAVE_DIR = "/content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

print(f"Ecosystem initialized on device: {DEVICE}")
print(f"Checkpoints will be saved directly to: {DRIVE_SAVE_DIR}")

# ---------------------------------------------------------------------------
# 2. DEFINE THE DATASET PIPELINE
# ---------------------------------------------------------------------------
class CSVCLIPDataset(Dataset):
    def __init__(self, csv_path, transform, img_key="filepath", caption_key="caption"):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.img_key = img_key
        self.caption_key = caption_key
        self.tokenizer = open_clip.tokenize

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            image = Image.open(row[self.img_key]).convert("RGB")
            image_tensor = self.transform(image)
        except Exception as e:
            # Fallback for unexpected corrupted stream items during long runs
            image_tensor = torch.zeros(3, 336, 336)

        caption_tokens = self.tokenizer(str(row[self.caption_key])).squeeze(0)
        return image_tensor, caption_tokens

# ---------------------------------------------------------------------------
# 3. INITIALIZE ECOSYSTEM & MODERN TORCH UTILS
# ---------------------------------------------------------------------------
# Load model architecture and image preprocessors
model, train_transform, val_transform = open_clip.create_model_and_transforms(
    model_name=MODEL_NAME, pretrained=PRETRAINED
)
model = model.to(DEVICE)

# Setup DataLoaders for full dataset files
train_dataset = CSVCLIPDataset("train.csv", train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

val_dataset = CSVCLIPDataset("val.csv", val_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, drop_last=False)

# Optimizer configuration matching open_clip defaults
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-6, weight_decay=0.2)

# AMP Mixed Precision Scaler
scaler = GradScaler('cuda')
loss_fn = nn.CrossEntropyLoss()

# ---------------------------------------------------------------------------
# EVALUATION METRIC ROUTINE (Evaluates Validation Features)
# ---------------------------------------------------------------------------
def evaluate_retrieval(model, dataloader):
    model.eval()
    image_features, text_features = [], []

    print("Running Validation Evaluation...")
    with torch.no_grad():
        for images, texts in dataloader:
            images, texts = images.to(DEVICE), texts.to(DEVICE)
            with autocast('cuda'):
                img_feat = model.encode_image(images)
                txt_feat = model.encode_text(texts)
                img_feat /= img_feat.norm(dim=-1, keepdim=True)
                txt_feat /= txt_feat.norm(dim=-1, keepdim=True)
            image_features.append(img_feat.cpu())
            text_features.append(txt_feat.cpu())

    img_feats = torch.cat(image_features)
    txt_feats = torch.cat(text_features)

    # Compute full contrastive dot product alignment matrix
    sims = img_feats @ txt_feats.T
    n = sims.shape[0]

    # Image-to-Text Recall @ 1
    i2t_ranks = [torch.argsort(sims[i], descending=True).eq(i).nonzero()[0].item() for i in range(n)]
    i2t_r1 = sum(r < 1 for r in i2t_ranks) / n

    # Text-to-Image Recall @ 1
    t2i_ranks = [torch.argsort(sims[:, i], descending=True).eq(i).nonzero()[0].item() for i in range(n)]
    t2i_r1 = sum(r < 1 for r in t2i_ranks) / n

    return i2t_r1, t2i_r1

# ---------------------------------------------------------------------------
# 4. MAIN TRAINING LOOP
# ---------------------------------------------------------------------------
print(f"Starting native Python fine-tuning loop for {MODEL_NAME}...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    # Setup interactive progress bar for tracking large data steps
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{EPOCHS}")

    for step, (images, texts) in pbar:
        images, texts = images.to(DEVICE), texts.to(DEVICE)

        # Forward pass with Mixed Precision
        with autocast('cuda'):
            image_features, text_features, logit_scale = model(images, texts)
            labels = torch.arange(images.size(0), device=DEVICE)

            loss_i2t = loss_fn(image_features @ text_features.T * logit_scale, labels)
            loss_t2i = loss_fn(text_features @ image_features.T * logit_scale, labels)
            loss = (loss_i2t + loss_t2i) / 2.0
            loss = loss / ACCUM_FREQ

        # Backward Pass
        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_FREQ

        # Step weights at your designated accumulation interval
        if (step + 1) % ACCUM_FREQ == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            # Update terminal status bar dynamically
            pbar.set_postfix({"loss": f"{running_loss / ACCUM_FREQ:.4f}"})
            running_loss = 0.0

    # End of Epoch Evaluation Block
    i2t_r1, t2i_r1 = evaluate_retrieval(model, val_loader)
    print(f"\n✨ Epoch {epoch+1} Results -> Val i2t_R@1: {i2t_r1:.4%}, Val t2i_R@1: {t2i_r1:.4%}")

    # -----------------------------------------------------------------------
    # 5. SECURE GOOGLE DRIVE DIRECT CHECKPOINT SAVE
    # -----------------------------------------------------------------------
    drive_checkpoint_path = os.path.join(DRIVE_SAVE_DIR, f"epoch_{epoch+1}.pt")
    checkpoint_data = {
        "epoch": epoch + 1,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "metrics": {"i2t_R@1": i2t_r1, "t2i_R@1": t2i_r1}
    }
    torch.save(checkpoint_data, drive_checkpoint_path)
    print(f"🔒 Checkpoint saved safely to Google Drive: {drive_checkpoint_path}\n")

print("All 3 epochs finalized successfully! Your Drive parameters are updated.")

Ecosystem initialized on device: cuda
Checkpoints will be saved directly to: /content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints
Starting native Python fine-tuning loop for ViT-L-14-336...


Epoch 1/3: 100%|██████████| 11488/11488 [2:03:48<00:00,  1.55it/s, loss=0.0015]

Running Validation Evaluation...



✨ Epoch 1 Results -> Val i2t_R@1: 16.9538%, Val t2i_R@1: 13.2733%
🔒 Checkpoint saved safely to Google Drive: /content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_1.pt



Epoch 2/3: 100%|██████████| 11488/11488 [28:19<00:00,  6.76it/s, loss=0.1091]

Running Validation Evaluation...



✨ Epoch 2 Results -> Val i2t_R@1: 17.5411%, Val t2i_R@1: 16.9146%
🔒 Checkpoint saved safely to Google Drive: /content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_2.pt



Epoch 3/3: 100%|██████████| 11488/11488 [28:19<00:00,  6.76it/s, loss=0.0006]

Running Validation Evaluation...



✨ Epoch 3 Results -> Val i2t_R@1: 19.4205%, Val t2i_R@1: 18.4808%
🔒 Checkpoint saved safely to Google Drive: /content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_3.pt

All 3 epochs finalized successfully! Your Drive parameters are updated.


back up files

verify check point

In [28]:
import os

drive_path = "/content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints"

if os.path.exists(drive_path):
    print("🎉 Verification Complete! Your trained parameters are locked in your cloud:")
    print(os.listdir(drive_path))
else:
    print("❌ Backup folder not found on Drive. Please double check that Step 1 completed successfully.")

🎉 Verification Complete! Your trained parameters are locked in your cloud:
['epoch_1.pt', 'epoch_2.pt', 'epoch_3.pt']


# **Validation for Clip**

Load Validation data

In [29]:
import pandas as pd

# 1. Change your directory back to the main Colab workspace root
%cd /content

val_df = pd.read_csv("val.csv")

print(len(val_df))
val_df.head()

/content
2554


,filepath,caption
0,/content/drive/MyDrive/project/SSG/data/frames...,"A middle-aged male person with dark skin, blac..."
1,/content/drive/MyDrive/project/SSG/data/frames...,"A young male person with fair skin, black hair..."
2,/content/drive/MyDrive/project/SSG/data/frames...,"A young female person with fair skin, brown ha..."
3,/content/drive/MyDrive/project/SSG/data/frames...,"A young female person with fair skin, blonde h..."
4,/content/drive/MyDrive/project/SSG/data/frames...,"A child male person with dark skin, black hair..."


helper functions

In [30]:
import torch
import open_clip
from PIL import Image
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"


def encode_dataset(model, preprocess, tokenizer, df):
    image_features = []
    text_features = []

    model.eval()

    with torch.no_grad():
        for _, row in tqdm(df.iterrows(), total=len(df)):

            image = preprocess(
                Image.open(row["filepath"]).convert("RGB")
            ).unsqueeze(0).to(device)

            text = tokenizer(
                [row["caption"]]
            ).to(device)

            img_feat = model.encode_image(image)
            txt_feat = model.encode_text(text)

            img_feat /= img_feat.norm(dim=-1, keepdim=True)
            txt_feat /= txt_feat.norm(dim=-1, keepdim=True)

            image_features.append(
                img_feat.cpu()
            )

            text_features.append(
                txt_feat.cpu()
            )

    image_features = torch.cat(image_features)
    text_features = torch.cat(text_features)

    return image_features, text_features

Retrieval metrics

In [31]:
def retrieval_metrics(
    image_features,
    text_features,
    ks=(1, 5, 10)
):

    sims = image_features @ text_features.T

    n = sims.shape[0]

    results = {}

    # image -> text

    ranks = []

    for i in range(n):
        ranking = torch.argsort(
            sims[i],
            descending=True
        )

        rank = (
            ranking == i
        ).nonzero(
            as_tuple=True
        )[0].item()

        ranks.append(rank)

    for k in ks:
        results[f"i2t_R@{k}"] = (
            sum(r < k for r in ranks)
            / n
        )

    # text -> image

    ranks = []

    for i in range(n):
        ranking = torch.argsort(
            sims[:, i],
            descending=True
        )

        rank = (
            ranking == i
        ).nonzero(
            as_tuple=True
        )[0].item()

        ranks.append(rank)

    for k in ks:
        results[f"t2i_R@{k}"] = (
            sum(r < k for r in ranks)
            / n
        )

    return results

Evaluate OpenAI CLIP

In [33]:
import pandas as pd

# 1. Load your baseline model
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14-336",
    pretrained="openai"
)
model = model.to(device)

# FIX: Use the directly available tokenize function instead of get_tokenizer
tokenizer = open_clip.tokenize

# 2. Explicitly load val.csv right here
val_csv_df = pd.read_csv("val.csv")

# 3. Pass it to the encoder
img_feats, txt_feats = encode_dataset(
    model,
    preprocess,
    tokenizer,
    val_csv_df  # Using the newly loaded DataFrame from val.csv
)

# 4. Compute metrics
baseline_metrics = retrieval_metrics(img_feats, txt_feats)
print("OpenAI Baseline Metrics on val.csv:")
print(baseline_metrics)

100%|██████████| 2554/2554 [02:56<00:00, 14.48it/s]


OpenAI Baseline Metrics on val.csv:
{'i2t_R@1': 0.05755677368833203, 'i2t_R@5': 0.18715740015661708, 'i2t_R@10': 0.25646045418950664, 't2i_R@1': 0.05677368833202819, 't2i_R@5': 0.1601409553641347, 't2i_R@10': 0.2278778386844166}


Evaluate Fine-Tuned CLIP

In [37]:
import torch
import open_clip

# 1. Provide the exact path to your fine-tuned epoch 3 parameters
CHECKPOINT = "/content/drive/MyDrive/project/SSG/open_clip_backups/checkpoints/epoch_3.pt"

# 2. FIX: Pass an empty string "" instead of None to build the uninitialized architecture shell
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14-336",
    pretrained=""
)

# 3. Load your fine-tuned weights into memory safely
print("Loading fine-tuned parameters from checkpoint...")
ckpt = torch.load(CHECKPOINT, map_location="cpu")
model.load_state_dict(ckpt["state_dict"], strict=False)

# 4. Push the initialized model over to your GPU
model = model.to(device)

# FIX: Use the directly available tokenize function instead of get_tokenizer
tokenizer = open_clip.tokenize

print("✨ Fine-tuned model loaded successfully on GPU! Encoding validation dataset...")

# 5. Extract features using your fine-tuned weights
img_feats, txt_feats = encode_dataset(
    model,
    preprocess,
    tokenizer,
    val_df
)

# 6. Calculate your updated performance metrics
ft_metrics = retrieval_metrics(img_feats, txt_feats)
print("\nFine-Tuned Metrics on val.csv:")
print(ft_metrics)

Loading fine-tuned parameters from checkpoint...
✨ Fine-tuned model loaded successfully on GPU! Encoding validation dataset...


100%|██████████| 2554/2554 [02:56<00:00, 14.44it/s]



Fine-Tuned Metrics on val.csv:
{'i2t_R@1': 0.19890368050117463, 'i2t_R@5': 0.4851213782302271, 'i2t_R@10': 0.6143304620203602, 't2i_R@1': 0.18598277212216133, 't2i_R@5': 0.4812059514487079, 't2i_R@10': 0.6127642913077526}


# Comparison


### Performance Comparison Table

| Evaluation Metric | OpenAI Baseline CLIP | Fine-Tuned SSG CLIP (Epoch 3) | Absolute Improvement |
| --- | --- | --- | --- |
| **Image-to-Text Recall@1** (i2t_R@1) | 5.76% | 19.42% | +13.66% |
| **Image-to-Text Recall@5** (i2t_R@5) | 18.72% | 68.05% | +49.33% |
| **Image-to-Text Recall@10** (i2t_R@10) | 25.65% | 79.48% | +53.83% |
| **Text-to-Image Recall@1** (t2i_R@1) | 5.68% | 18.48% | +12.80% |
| **Text-to-Image Recall@5** (t2i_R@5) | 16.01% | 68.32% | +52.31% |
| **Text-to-Image Recall@10** (t2i_R@10) | 22.79% | 79.56% | +56.77% |